# Загружаем данные из API LSEG REFINITIV

In [1]:
import os
from datetime import datetime, timezone
import pandas as pd

import lseg.data as ld

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
FOLDER_PATH = r"C:\Courses\Innopolis\2026\ИИ Инженер\Data\Spark"

In [3]:
os.environ["NO_PROXY"] = "localhost"

In [4]:
ld.open_session()

<lseg.data.session.Definition object at 0x22134074ec0 {name='workspace'}>

In [6]:
# Список инструментов
instruments = [
    "AAPL.O",
    "MSFT.O",
    "NVDA.O",
    "AMZN.O"
]

# Получаем исторические данные
response = ld.get_history(
    universe=instruments,
    fields=[
        "TR.PriceOpen",
        "TR.PriceHigh",
        "TR.PriceLow",
        "TR.PriceClose",
        "TR.Volume"
    ],
    interval="daily",
    start="2026-01-01",
    end="2026-11-06"
)

In [7]:
response.head()

AAPL.O                                                MSFT.O  \
           Price Open Price High Price Low Price Close    Volume Price Open   
Date                                                                          
2026-01-02    272.255     277.84     269.0      271.01  37838054    484.385   
2026-01-05     270.64     271.51    266.14      267.26  45647190    474.055   
2026-01-06      267.0     267.55    262.12      262.36  52352090      473.8   
2026-01-07      263.2     263.68    259.81      260.33  48309804    479.755   
2026-01-08     257.02     259.29     255.7      259.04  50419337     481.24   

                                                          NVDA.O             \
           Price High Price Low Price Close    Volume Price Open Price High   
Date                                                                          
2026-01-02     484.66    470.16      472.94  25571567    189.835   192.9299   
2026-01-05     476.07     469.5      472.85  25250260     191.76     193.63   
2026-01-06   478.7367    469.75      478.51  23037665     190.52    192.174   
2026-01-07      489.7    477.95      483.47  25564196     188.57     191.37   
2026-01-08     482.66    475.86      478.11  18162616     189.11     189.55   

                                                AMZN.O                       \
           Price Low Price Close     Volume Price Open Price High Price Low   
Date                                                                          
2026-01-02    188.26      188.85  148240462     231.34    235.458     224.7   
2026-01-05    186.15      188.12  183529733     228.84      234.0    227.18   
2026-01-06    186.82      187.24  176862604      232.1     243.18    232.07   
2026-01-07    186.56      189.11  153543174     239.61     245.29    239.52   
2026-01-08    183.71      185.04  172457048     243.06     246.41    241.88   

                                  
           Price Close    Volume  
Date                              
2026-01-02       226.5  51456229  
2026-01-05      233.06  49733348  
2026-01-06      240.93  53764677  
2026-01-07      241.56  42236531  
2026-01-08      246.29  39509844

In [7]:
# Закрываем сессию
ld.close_session()

# Bronze

In [8]:
df = response.swaplevel(axis=1).stack().reset_index()
df.columns = ["date", "instrument", "price_open", "price_high", "price_low", "price_close", "volume"]

In [9]:
df.head()

,date,instrument,price_open,price_high,price_low,price_close,volume
0,2026-01-02,AAPL.O,272.255,277.84,269.0,271.01,37838054
1,2026-01-02,AMZN.O,231.34,235.458,224.7,226.5,51456229
2,2026-01-02,MSFT.O,484.385,484.66,470.16,472.94,25571567
3,2026-01-02,NVDA.O,189.835,192.9299,188.26,188.85,148240462
4,2026-01-05,AAPL.O,270.64,271.51,266.14,267.26,45647190


In [11]:
# Предполагаем, что df содержит нормализованный результат получения данных из LSEG
bronze_df = df.copy()

# Метаданные загрузки
bronze_df["source"] = "LSEG_REFINITIV"
bronze_df["ingestion_time"] = pd.to_datetime(datetime.now(timezone.utc))
bronze_df["batch_id"] = "batch_001"

# Сохраняем исходный набор
bronze_df.to_csv(FOLDER_PATH + r"\bronze\lseg_prices_quality.csv", index=False)

# Проверка полноты (completeness)

In [12]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [13]:
spark = SparkSession.builder.appName("LSEG_Data_Architecture").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

In [14]:
bronze = spark.read.csv(FOLDER_PATH + r"\bronze\lseg_prices_quality.csv", header=True, inferSchema=True)

In [16]:
required_columns = ["date", "instrument", "price_open", "price_high", "price_low", "price_close", "volume"]

completeness = (
    bronze
    .select(
        [
            (F.sum(F.when(F.col(c).isNotNull(), F.lit(1)).otherwise(F.lit(0))) / F.count("*") * 100).alias(c + "_null_pct") for c in required_columns
        ]
    )
)

completeness.show()

+-------------+-------------------+-------------------+-------------------+------------------+--------------------+---------------+
|date_null_pct|instrument_null_pct|price_open_null_pct|price_high_null_pct|price_low_null_pct|price_close_null_pct|volume_null_pct|
+-------------+-------------------+-------------------+-------------------+------------------+--------------------+---------------+
|        100.0|              100.0|              100.0|              100.0|             100.0|               100.0|          100.0|
+-------------+-------------------+-------------------+-------------------+------------------+--------------------+---------------+



# Проверка допустимости значений (validity)

In [22]:
validity = (
    bronze
    .withColumn(
        "valid_prices",
        (F.col("price_open") > 0) &
        (F.col("price_high") > 0) &
        (F.col("price_low") > 0) &
        (F.col("price_close") > 0)
    )
    .withColumn(
        "valid_ohlc",
        (F.col("price_high") >= F.col("price_low")) &
        (F.col("price_open") >= F.col("price_low")) &
        (F.col("price_open") <= F.col("price_high")) &
        (F.col("price_close") >= F.col("price_low")) &
        (F.col("price_close") <= F.col("price_high"))
    )
    .withColumn(
        "valid_volume",
        F.col("volume").isNull() |
        (F.col("volume") >= 0)
    )
)

(
    validity
    .select(
        F.min(F.col("valid_prices")).alias("valid_prices"),
        F.min(F.col("valid_ohlc")).alias("valid_ohlc"),
        F.min(F.col("valid_volume")).alias("valid_volume")
    )
).show()

+------------+----------+------------+
|valid_prices|valid_ohlc|valid_volume|
+------------+----------+------------+
|        true|      true|        true|
+------------+----------+------------+



# Проверка уникальности (uniqueness)

In [23]:
duplicates = (
    bronze
    .groupBy(
        F.col("instrument"),
        F.col("date")
    )
    .agg(
        F.count("*").alias("records_count")
    )
    .filter(
        F.col("records_count") > 1
    )
)

duplicates.show()

+----------+----+-------------+
|instrument|date|records_count|
+----------+----+-------------+
+----------+----+-------------+



# Проверка аномальных изменений цен (accuracy)

In [27]:
window = (
    Window
    .partitionBy("instrument")
    .orderBy("date")
)

price_changes = (
    bronze
    .dropDuplicates(["instrument", "date"])
    .withColumn("previous_close", F.lag("price_close").over(window))
    .withColumn("daily_return", F.col("price_close") / F.col("previous_close") - 1)
    .withColumn("is_anomaly", F.abs(F.col("daily_return")) > 0.20) # По опыту, изменение цены на 20% и более считается аномальным
)

(
    price_changes
    .where(F.col("is_anomaly") == True)
    .select(
        F.col("instrument"),
        F.col("date"),
        F.col("price_close"),
        F.col("previous_close"),
        F.col("daily_return"),
        F.col("is_anomaly")
    )
).show()

+----------+----+-----------+--------------+------------+----------+
|instrument|date|price_close|previous_close|daily_return|is_anomaly|
+----------+----+-----------+--------------+------------+----------+
+----------+----+-----------+--------------+------------+----------+



# Проверка согласованности и временной последовательности

In [29]:
instrument_stats = (
    bronze
    .groupBy("instrument")
    .agg(
        F.min("date").alias("first_date"),
        F.max("date").alias("last_date"),
        F.countDistinct("date").alias("trading_days"),
        F.count("*").alias("total_records")
    )
)

instrument_stats.show()

+----------+----------+----------+------------+-------------+
|instrument|first_date| last_date|trading_days|total_records|
+----------+----------+----------+------------+-------------+
|    AAPL.O|2026-01-02|2026-11-06|         214|          214|
|    AMZN.O|2026-01-02|2026-11-06|         214|          214|
|    MSFT.O|2026-01-02|2026-11-06|         214|          214|
|    NVDA.O|2026-01-02|2026-11-06|         214|          214|
+----------+----------+----------+------------+-------------+

